# BLSTM Translator (Amharic/Ge'ez/English) — Colab Notebook
Robust, trainable seq2seq with **Bidirectional LSTM encoder + attention decoder**. Includes data loading, training, checkpointing, evaluation (BLEU), and batch inference. Works with CSV columns like `amh, gez, eng`.

**Tip:** Runtime → Change runtime type → GPU.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
#@title Setup (installs & imports)
!pip -q install tensorflow==2.19.0 sacrebleu==2.4.0 pandas==2.2.2 numpy==1.26.4 --no-warn-script-location
import os, math, random, json, pickle, gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, AdditiveAttention, Concatenate, TimeDistributed, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras import mixed_precision
from google.colab import files
import sacrebleu

print('TensorFlow:', tf.__version__)
try:
    from tensorflow.python.client import device_lib
    print('Devices:', [d.name for d in device_lib.list_local_devices()])
except Exception as e:
    print('Device check skipped:', e)

# Optional: enable mixed precision for speed on GPUs with Tensor Cores
USE_MIXED_PRECISION = True  #@param {type:"boolean"}
if USE_MIXED_PRECISION:
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("Mixed precision:", mixed_precision.global_policy())
else:
    print("Mixed precision: disabled")

# Optional: gradient clipping
CLIP_NORM = 1.0  #@param {type:"number"}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.3/106.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 123.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompat

## Upload your CSV
Upload a CSV with columns like `amh, gez, eng`. You can train any direction by choosing `src_col` and `tgt_col` below.

In [6]:

#@title Upload CSV
csv_path = "/content/drive/MyDrive/geez-amharic-eng-translator/AGE.csv"
print("Using:", csv_path)


Using: /content/drive/MyDrive/geez-amharic-eng-translator/AGE.csv


## Configuration
Tune training lengths, epochs, and vocab sizes to fit your GPU/time.

In [7]:

#@title Config
src_col = "gez"  #@param {type:"string"}
tgt_col = "amh"  #@param {type:"string"}
sample_frac = None  #@param {type:"number"}
epochs = 8  #@param {type:"integer"}
batch_size = 64  #@param {type:"integer"}
max_src_len = 80  #@param {type:"integer"}
max_tgt_len = 80  #@param {type:"integer"}
src_vocab = 20000  #@param {type:"integer"}
tgt_vocab = 20000  #@param {type:"integer"}
val_split = 0.1  #@param {type:"number"}
dropout = 0.2  #@param {type:"number"}

out_dir = "artifacts"
os.makedirs(out_dir, exist_ok=True)

def seed_all(seed=42):
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)

seed_all(42)


## Load & Preview Data

In [8]:

#@title Load & peek
def read_data(csv_path, src_col, tgt_col, sample_frac=None):
    df = pd.read_csv(csv_path)
    if src_col not in df.columns or tgt_col not in df.columns:
        raise ValueError(f"CSV must contain columns '{src_col}' and '{tgt_col}'. Found: {list(df.columns)}")
    df = df[[src_col, tgt_col]].dropna()
    if sample_frac is not None and 0 < sample_frac < 1.0:
        df = df.sample(frac=sample_frac, random_state=42)
    df[src_col] = df[src_col].astype(str).str.strip()
    df[tgt_col] = df[tgt_col].astype(str).str.strip()
    return df.reset_index(drop=True)

df = read_data(csv_path, src_col, tgt_col, sample_frac=sample_frac)
print(f"Pairs: {len(df)}")
display(df.head(5))


Pairs: 17503


,gez,amh
0,ወዳዊትሰ ንጉሥ ልህቀ ወኀለፈ መዋዕሊሁ ወይከድንዎ አልባሰ ወኢያመውቆ ።,ንጉሡ ዳዊትም ሸመገለ ዕድሜውም በዛ፤ ልብስም ደረቡለት፥ ነገር ግን አይሞ...
1,ወይቤሉ ደቁ ለዳዊት ይኅሥሡ ለእግዚእነ ለንጉሥ ወለተ ድንግለ ወያምጽእዋ ...,ባሪያዎቹም። ለጌታችን ለንጉሡ ድንግል ትፈለጋለች፤ በንጉሡም ፊት ቆማ ታገ...
2,ወኀሠሡ ወለተ ሠናይተ እምነ ኵሉ ደወለ እስራኤል ወረከቡ አቢሳሃ ሰሜናዊት...,በእስራኤልም አገር ሁሉ የተዋበች ቈንጆ ፈለጉ፤ ሱነማይቱን አቢሳንም አገኙ...
3,ወነበረት ትትለአኮ ወንጉሥሰ ኢያእመራ ።,ቈንጆይቱም እጅግ ውብ ነበረች፤ ንጉሡንም ትረዳውና ታገለግለው ነበር፥ ንጉ...
4,ወአዶንያስ ወልደ አጊት ተንሥአ ወይቤ አነ እነግሥ ወገብረ ሎቱ ሰረገላተ ...,የአጊትም ልጅ አዶንያስ። ንጉሥ እሆናለሁ ብሎ ተነሣ፤ ሰረገሎችንና ፈረሰኞ...


## Tokenization
Whitespace tokenization that preserves Ethiopic scripts. We add `<s>` and `</s>` around the target for teacher forcing.

In [9]:

#@title Tokenizers
def prepare_tokenizers(src_texts, tgt_texts, num_words_src=20000, num_words_tgt=20000, oov_token="<unk>"):
    src_tok = Tokenizer(num_words=num_words_src, filters="", lower=False, oov_token=oov_token, split=" ")
    tgt_tok = Tokenizer(num_words=num_words_tgt, filters="", lower=False, oov_token=oov_token, split=" ")
    tgt_in_texts  = [f"<s> {t}" for t in tgt_texts]
    tgt_out_texts = [f"{t} </s>" for t in tgt_texts]
    src_tok.fit_on_texts(src_texts)
    tgt_tok.fit_on_texts(tgt_in_texts + tgt_out_texts)
    return src_tok, tgt_tok, tgt_in_texts, tgt_out_texts

src_tok, tgt_tok, tgt_in_texts, tgt_out_texts = prepare_tokenizers(
    df[src_col].tolist(), df[tgt_col].tolist(), src_vocab, tgt_vocab
)

# dynamic lengths (bounded by user caps)
dyn_max_src = min(max_src_len, max((len(s.split()) for s in df[src_col]), default=1))
dyn_max_tgt = min(max_tgt_len, max((len(s.split())+2 for s in df[tgt_col]), default=1))
dyn_max_src = max(dyn_max_src, 4)
dyn_max_tgt = max(dyn_max_tgt, 4)

def texts_to_padded(tokenizer, texts, maxlen):
    seqs = tokenizer.texts_to_sequences(texts)
    return pad_sequences(seqs, maxlen=maxlen, padding="post", truncating="post")

X_src = texts_to_padded(src_tok, df[src_col].tolist(), dyn_max_src)
X_tgt_in = texts_to_padded(tgt_tok, tgt_in_texts, dyn_max_tgt)
X_tgt_out = texts_to_padded(tgt_tok, tgt_out_texts, dyn_max_tgt)
y = np.expand_dims(X_tgt_out, -1)

print("dyn_max_src:", dyn_max_src, "dyn_max_tgt:", dyn_max_tgt)
print("src vocab size:", min(src_vocab, len(src_tok.word_index)+1))
print("tgt vocab size:", min(tgt_vocab, len(tgt_tok.word_index)+1))


dyn_max_src: 77 dyn_max_tgt: 55
src vocab size: 20000
tgt vocab size: 20000


## Model: BLSTM Encoder + Attention Decoder

In [19]:
#@title Build model (disable auto-masking + custom Bahdanau attention)
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, TimeDistributed, Dropout, Concatenate, Lambda
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer
import tensorflow as tf

class NoMask(Layer):
    def call(self, x):
        return x
    def compute_mask(self, inputs, mask=None):
        # Explicitly drop mask so nothing downstream receives it
        return None

def build_model(
    src_vocab_size, tgt_vocab_size,
    emb_dim=256, enc_units=256, dec_units=256,
    max_src_len=80, max_tgt_len=80, dropout=0.2, clip_norm=1.0
):
    # Inputs (0 = PAD id)
    src_in = Input(shape=(max_src_len,), name="src_in")
    tgt_in = Input(shape=(max_tgt_len,), name="tgt_in")

    # Embeddings: TURN OFF mask_zero to prevent Keras mask propagation.
    enc_emb = Embedding(src_vocab_size, emb_dim, mask_zero=False, name="src_emb")(src_in)
    dec_emb = Embedding(tgt_vocab_size, emb_dim, mask_zero=False, name="tgt_emb")(tgt_in)

    # Encoder: BiLSTM -> (B, T_k, 2*enc_units)
    enc_blstm, f_h, f_c, b_h, b_c = Bidirectional(
        LSTM(enc_units, return_sequences=True, return_state=True, name="enc_lstm"),
        name="bilstm"
    )(enc_emb)

    # Bridge encoder states → decoder init (dec_units)
    state_h = Concatenate(name="enc_state_h")([f_h, b_h])
    state_c = Concatenate(name="enc_state_c")([f_c, b_c])
    state_h = Dense(dec_units, activation="tanh", name="map_h")(state_h)
    state_c = Dense(dec_units, activation="tanh", name="map_c")(state_c)

    # Decoder outputs (queries): (B, T_q, dec_units)
    dec_out, _, _ = LSTM(dec_units, return_sequences=True, return_state=True, name="dec_lstm")(
        dec_emb, initial_state=[state_h, state_c]
    )

    # ----- Custom Bahdanau attention (explicit masks we control) -----
    # Project to dec_units
    Wq = TimeDistributed(Dense(dec_units, use_bias=False), name="Wq")(dec_out)      # (B, T_q, D)
    Wk = TimeDistributed(Dense(dec_units, use_bias=False), name="Wk")(enc_blstm)    # (B, T_k, D)

    # Broadcast to (B, T_q, T_k, D)
    Wq_e = Lambda(lambda x: tf.expand_dims(x, axis=2), name="expand_Wq")(Wq)
    Wk_e = Lambda(lambda x: tf.expand_dims(x, axis=1), name="expand_Wk")(Wk)

    e_tanh = Lambda(lambda x: tf.tanh(x[0] + x[1]), name="e_tanh")([Wq_e, Wk_e])    # (B, T_q, T_k, D)
    e = TimeDistributed(TimeDistributed(Dense(1, use_bias=True)), name="score")(e_tanh)
    e = Lambda(lambda x: tf.squeeze(x, axis=-1), name="score_squeeze")(e)           # (B, T_q, T_k)

    # Our own masks (True = real token)
    tgt_mask = Lambda(lambda x: tf.not_equal(x, 0), name="tgt_mask")(tgt_in)        # (B, T_q)
    src_mask = Lambda(lambda x: tf.not_equal(x, 0), name="src_mask")(src_in)        # (B, T_k)

    # Apply source mask to scores (pad -> -inf)
    def _apply_src_mask(args):
        scores, mask = args  # scores: (B, T_q, T_k), mask: (B, T_k)
        mask = tf.cast(mask, scores.dtype)
        mask = tf.expand_dims(mask, axis=1)                     # (B, 1, T_k)
        neg_inf = tf.constant(-1e9, dtype=scores.dtype)
        return scores + (1.0 - mask) * neg_inf

    e_masked = Lambda(_apply_src_mask, name="apply_src_mask")([e, src_mask])

    # Softmax over keys
    alphas = Lambda(lambda x: tf.nn.softmax(x, axis=-1), name="alphas")(e_masked)   # (B, T_q, T_k)

    # Context = alphas @ values
    context = Lambda(lambda x: tf.matmul(x[0], x[1]), name="context")([alphas, enc_blstm])  # (B, T_q, 2*enc_units)

    # Optionally zero out context at padded target positions
    def _mask_context(args):
        ctx, qmask = args
        qmask = tf.cast(qmask, ctx.dtype)
        qmask = tf.expand_dims(qmask, axis=-1)                  # (B, T_q, 1)
        return ctx * qmask

    context = Lambda(_mask_context, name="mask_context")([context, tgt_mask])

    # Combine and predict
    comb = Concatenate(name="attn_concat")([dec_out, context])  # (B, T_q, dec_units + 2*enc_units)
    comb = Dropout(dropout, name="dropout")(comb)

    # Drop any residual masks explicitly, then project
    comb = NoMask(name="drop_mask")(comb)
    logits = TimeDistributed(Dense(tgt_vocab_size, activation="softmax", dtype="float32"), name="logits")(comb)

    model = Model([src_in, tgt_in], logits)
    opt = tf.keras.optimizers.Adam(learning_rate=3e-4, clipnorm=clip_norm if clip_norm else None)
    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


In [23]:
model = build_model(
    src_vocab_size=min(src_vocab, len(src_tok.word_index)+1),
    tgt_vocab_size=min(tgt_vocab, len(tgt_tok.word_index)+1),
    emb_dim=256, enc_units=256, dec_units=256,
    max_src_len=dyn_max_src, max_tgt_len=dyn_max_tgt,
    dropout=dropout, clip_norm=CLIP_NORM
)
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ src_in (InputLayer) │ (None, 77)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ src_emb (Embedding) │ (None, 77, 256)   │  5,120,000 │ src_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bilstm              │ [(None, 77, 512), │  1,050,624 │ src_emb[0][0]     │
│ (Bidirectional)     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tgt_in (InputLayer) │ (None, 55)        │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_state_h         │ (None, 512)       │          0 │ bilstm[0][1],     │
│ (Concatenate)       │                   │            │ bilstm[0][3]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_state_c         │ (None, 512)       │          0 │ bilstm[0][2],     │
│ (Concatenate)       │                   │            │ bilstm[0][4]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tgt_emb (Embedding) │ (None, 55, 256)   │  5,120,000 │ tgt_in[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ map_h (Dense)       │ (None, 256)       │    131,328 │ enc_state_h[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ map_c (Dense)       │ (None, 256)       │    131,328 │ enc_state_c[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm (LSTM)     │ [(None, 55, 256), │    525,312 │ tgt_emb[0][0],    │
│                     │ (None, 256),      │            │ map_h[0][0],      │
│                     │ (None, 256)]      │            │ map_c[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wq                  │ (None, 55, 256)   │     65,536 │ dec_lstm[0][0]    │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Wk                  │ (None, 77, 256)   │    131,072 │ bilstm[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_Wq (Lambda)  │ (None, 55, 1,     │          0 │ Wq[0][0]          │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_Wk (Lambda)  │ (None, 1, 77,     │          0 │ Wk[0][0]          │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ e_tanh (Lambda)     │ (None, 55, 77,    │          0 │ expand_Wq[0][0],  │
│                     │ 256)              │            │ expand_Wk[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score               │ (None, 55, 77, 1) │        257 │ e_tanh[0][0]      │
│ (TimeDistributed)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_squeeze       │ (None, 55, 77)    │          0 │ score[0][0]       │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 27,655,457 (105.50 MB)

 Trainable params: 27,655,457 (105.50 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# ensure integer type and proper shape
y = np.expand_dims(X_tgt_out, -1).astype("int32")  # or int64


In [22]:
opt = tf.keras.optimizers.Adam(learning_rate=3e-4, clipnorm=CLIP_NORM)
# or lower LR slightly


## Train
Early stopping + model checkpointing (best weights).

In [ ]:

#@title Train
ckpt_path = os.path.join(out_dir, "best_model.keras")
callbacks=[
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor="val_loss", save_best_only=True, save_weights_only=False)
]
history = model.fit(
    [X_src, X_tgt_in], y,
    validation_split=val_split,
    batch_size=batch_size,
    epochs=epochs,
    callbacks=callbacks,
    verbose=2
)
print("Best checkpoint saved to:", ckpt_path)


Epoch 1/8


## Save Tokenizers + Meta

In [ ]:

#@title Save artifacts
with open(os.path.join(out_dir, "src_tokenizer.pkl"), "wb") as f:
    pickle.dump(src_tok, f)
with open(os.path.join(out_dir, "tgt_tokenizer.pkl"), "wb") as f:
    pickle.dump(tgt_tok, f)
meta = {"src_col": src_col, "tgt_col": tgt_col, "max_src_len": int(dyn_max_src), "max_tgt_len": int(dyn_max_tgt)}
with open(os.path.join(out_dir, "meta.json"), "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)
print("Artifacts saved in:", out_dir)


## Greedy Decoder (Inference)

In [ ]:
#@title Greedy decode helpers
def build_encoder_subgraph(model):
    src_input_layer = model.get_layer("src_in")
    src_shape = src_input_layer.input_shape[1:]
    src_in = tf.keras.Input(shape=src_shape, dtype="int32", name="enc_src_in")
    enc_emb = model.get_layer("src_emb")(src_in)
    bilstm = model.get_layer("bilstm")
    enc_blstm, f_h, f_c, b_h, b_c = bilstm(enc_emb)
    state_h = model.get_layer("enc_state_h")([f_h, b_h])
    state_c = model.get_layer("enc_state_c")([f_c, b_c])
    map_h = model.get_layer("map_h")(state_h)
    map_c = model.get_layer("map_c")(state_c)
    return tf.keras.Model(src_in, [enc_blstm, map_h, map_c], name="encoder_subgraph")

def greedy_decode(model, src_seq, src_tok, tgt_tok, max_src_len, max_tgt_len):
    if len(src_seq) == 0:
        src_seq = [src_tok.word_index.get("<unk>", 0)]
    enc_model = build_encoder_subgraph(model)
    src_seq = pad_sequences([src_seq], maxlen=max_src_len, padding="post", dtype="int32")
    enc_outputs, h, c = enc_model.predict(src_seq, verbose=0)

    dec_emb_layer = model.get_layer("tgt_emb")
    dec_lstm = model.get_layer("dec_lstm")
    attn = model.get_layer("attention")
    attn_concat = model.get_layer("attn_concat")
    dropout = model.get_layer("dropout")
    logits_td = model.get_layer("logits")

    index2word = {i:w for w,i in tgt_tok.word_index.items()}
    index2word[0] = "<pad>"
    start_id = tgt_tok.word_index.get("<s>")
    end_id = tgt_tok.word_index.get("</s>")
    if start_id is None or end_id is None:
        raise RuntimeError("Missing <s> or </s> in target tokenizer.")

    y = np.array([[start_id]], dtype="int32")
    out_tokens = []
    for _ in range(max_tgt_len):
        y_emb = dec_emb_layer(y)
        dec_out, h, c = dec_lstm(y_emb, initial_state=[h, c])
        context = attn([dec_out, enc_outputs])
        comb = attn_concat([dec_out, context])
        comb = dropout(comb, training=False)
        probs = logits_td(comb).numpy()
        next_id = int(np.argmax(probs[0,0]))
        if next_id == end_id:
            break
        out_tokens.append(next_id)
        y = np.array([[next_id]], dtype="int32")
    out_words = [index2word.get(i, "<unk>") for i in out_tokens]
    return " ".join(out_words)

def translate_sentences(model, sentences, src_tok, tgt_tok, max_src_len, max_tgt_len):
    results = []
    for s in sentences:
        seqs = src_tok.texts_to_sequences([s])
        seq = seqs[0] if seqs else []
        pred = greedy_decode(model, seq, src_tok, tgt_tok, max_src_len, max_tgt_len)
        results.append(pred)
    return results


## Quick Sanity Translations

In [ ]:

#@title Sample predictions
samples = df[src_col].tolist()[:5]
preds = translate_sentences(model, samples, src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)
for s,p,t in zip(samples, preds, df[tgt_col].tolist()[:5]):
    print("SRC:", s)
    print("PRED:", p)
    print("TGT:", t)
    print("-"*80)


## BLEU Evaluation
On a small held-out split (random 200 by default).

In [ ]:

#@title Evaluate BLEU
eval_n = 200  #@param {type:"integer"}
idx = np.random.choice(len(df), size=min(eval_n, len(df)), replace=False)
refs = []
hyps = []
for i in idx:
    s = df.iloc[i][src_col]
    t = df.iloc[i][tgt_col]
    pred = translate_sentences(model, [s], src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)[0]
    refs.append(t)
    hyps.append(pred)
# SacreBLEU expects list of hypothesis strings and list of reference-list(s)
bleu = sacrebleu.corpus_bleu(hyps, [refs])
print("BLEU:", bleu.score)


## Batch Translate a File
Upload a TXT with one source sentence per line.

In [ ]:

#@title Batch translate (TXT → CSV)
from datetime import datetime

print("Upload a .txt with one sentence per line...")
uploaded_txt = files.upload()
txt_path = list(uploaded_txt.keys())[0]
with open(txt_path, "r", encoding="utf-8") as f:
    src_lines = [line.strip() for line in f if line.strip()]

hyps = translate_sentences(model, src_lines, src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)
out_csv = f"translations_{src_col}_to_{tgt_col}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
pd.DataFrame({src_col: src_lines, tgt_col: hyps}).to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv)


## Reload Saved Model
Load from checkpoint and reuse the same tokenizers.

In [ ]:

#@title Reload best checkpoint
reloaded = tf.keras.models.load_model(os.path.join(out_dir, "best_model.keras"))
print("Reloaded.")
# Sanity test
test_src = df[src_col].iloc[0]
print("SRC:", test_src)
print("PRED:", translate_sentences(reloaded, [test_src], src_tok, tgt_tok, dyn_max_src, dyn_max_tgt)[0])


## Tips for speed/robustness
- Use `sample_frac=0.3` while iterating.
- Reduce `epochs` to fit your time budget; add back later.
- Increase `batch_size` if you have more GPU memory.
- Mixed precision is enabled by default.
- Gradient clipping (`CLIP_NORM`) helps stabilize training.
- Use the checkpoint `.keras` file for the best model at inference time.